# Milestone 2 RAG Implementation

In [3]:
import transformers
from transformers import pipeline
import torch

import os
from pathlib import Path
import pandas as pd
import duckdb
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
import pickle
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever



In [4]:
if Path.cwd().name != "DSCI_575_project_jt8919_nicolelink33":
    project_root = Path.cwd().parent 
    os.chdir(project_root)
    
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/jennifertsang/Documents/MDS/DSCI 575 Advanced ML/DSCI_575_project_jt8919_nicolelink33


In [5]:
from src.utils import simple_tokenize, display_results
from src.bm25 import search_bm25_with_scores
from src.semantic import search_semantic_with_scores

In [6]:
DATA_DIR = Path("data")
CATEGORY = "Arts_Crafts_and_Sewing"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"
OUTPUT_FILE  = DATA_DIR / f"{CATEGORY}_merged.parquet"
bm25_path = "models/bm25_metadata_index_2.pkl"
semantic_path = "models/faiss_index"
OUTPUT = 'data/processed/stratified_sample.parquet'

## Setup
- Create LangChain Documents using product title, category, and review text of the product as the `page_content`, the rest of the data are stored in the `metadata` dictionary

In [7]:
# Load stratified sample
df_sample = pd.read_parquet(OUTPUT)

# Fill any random missing text values to prevent concatenation crashes
text_columns = ['product_title', 'main_category', 'text']
for col in text_columns:
    if col in df_sample.columns:
        df_sample[col] = df_sample[col].fillna("")

# Engineer the hybrid page_content string
df_sample['page_content'] = (
    "Product Title: " + df_sample['product_title'] + "\n" +
    "Category: " + df_sample['main_category'] + "\n" +
    "Review Text: " + df_sample['text']
)

# Drop the redundant text columns so they don't clutter the metadata
df_clean = df_sample.drop(columns=['product_title', 'main_category', 'text'])

# Convert the DataFrame into LangChain Documents
documents = []
for _, row in df_clean.iterrows():
    # The engineered string becomes the searchable content
    content = row['page_content']
    
    # Every other column (rating, price, helpful_vote, rating_bucket, len_tier) 
    # gets scooped up into the metadata dictionary!
    metadata = row.drop('page_content').to_dict()
    
    # Create and append the Document
    doc = Document(page_content=content, metadata=metadata)
    documents.append(doc)

print(f"Successfully created {len(documents)} LangChain Documents!")
print("\n--- Let's peek at the first one ---")
print(f"CONTENT:\n{documents[0].page_content}")
print(f"METADATA:\n{documents[0].metadata}")

Successfully created 641 LangChain Documents!

--- Let's peek at the first one ---
CONTENT:
Product Title: Leather Repair Patch Tape Kit Coffee Dark Brown 4 x 60 inch Self Adhesive Leather Repair Patch for Furniture, Couch, Sofa, Car Seats, Office Chair Vinyl Repair Kit
Category: Industrial & Scientific
Review Text: Didn't adhere on lawn mower seat very well. Came loose after one week.
METADATA:
{'rating': 3.0, 'verified_purchase': True, 'helpful_vote': 1, 'parent_asin': 'B09L1JT7SV', 'price': 11.68, 'average_rating': 4.0, 'rating_bucket': '3.7-4.0', 'len_tier': 'short'}


### Picking a model:
- Chose a `HuggingFace model`: `Qwen/Qwen2.5-0.5B`

In [8]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B",
    torch_dtype = torch.float16,
    device="mps",
    trust_remote_code=True,
    max_new_tokens=128,
    do_sample=True,
)

llm = HuggingFacePipeline(pipeline=generator)

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use mps


## RAG Explorations

#### With Semantic Retriever

In [9]:
# Loading in saved vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(documents, embeddings)

In [10]:
# Convert vector store to a retriever
semantic_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5} # returns top 5 documents
    )

# Use in a chain
query = "Green yarn"
docs = semantic_retriever.invoke(query)
docs

[Document(id='728046f8-b74b-4e7d-96f6-970cff37be31', metadata={'rating': 5.0, 'verified_purchase': True, 'helpful_vote': 5, 'parent_asin': 'B0BWSF1HPY', 'price': 33.99, 'average_rating': 4.1, 'rating_bucket': '4.1-4.3', 'len_tier': 'medium'}, page_content='Product Title: Arm Knitting Yarn for Chunky Braided Knot Throw Blanket DIY, Soft Extra Cotton Washable Tube Bulky Giant Yarn for Weave Craft Crochet (Dark Gray 2.2lb)\nCategory: Amazon Home\nReview Text: [[VIDEOID:21dfc200bc65819023d8994064a70121]] Seems like simple and normal yarn.  The feel was pretty soft I thought.<br /><br />Color seems accurate to the pictures.  Felt like what you’d use to make sweaters or something similar.'),
 Document(id='11130dab-4092-4323-9395-bb2ef2bcd991', metadata={'rating': 1.0, 'verified_purchase': True, 'helpful_vote': 23, 'parent_asin': 'B07YF38F25', 'price': 13.97, 'average_rating': 4.7, 'rating_bucket': '4.6-5.0', 'len_tier': 'long'}, page_content='Product Title: Lion Brand Yarn Ferris Wheel Yarn,

### Context Building
- Convert retrieved documents into a prompt-ready context block

In [11]:
def build_context(docs):
    context_parts = []
    for doc in docs:
        content = doc.page_content
        
        # 1. Extract the Title
        try:
            title = content.split("Product Title: ")[1].split("Category: ")[0].strip()
        except IndexError:
            title = "Unknown Product"

        # 2. Extract the Review Text 
        # This takes everything appearing after "Review Text: "
        try:
            review = content.split("Review Text: ")[1].strip()
        except IndexError:
            review = "No review text available."

        # 3. Assemble the formatted string
        entry = (
            f"Product ASIN: {doc.metadata.get('parent_asin', 'N/A')}\n"
            f"Title: {title}\n"
            f"Rating: {doc.metadata.get('rating', 'N/A')}/5\n"
            f"Review: {review}"
        )
        context_parts.append(entry)
    
    return "\n\n".join(context_parts)

### Prompt Template Design
- Defining a function directly within the RAG pipeline script
- Experiment with 2 different system prompt variants to find out which one works better

Considerations when designing the prompt:
- Clearly define the task
- Restrict the model to using only the provided context
- Encourage concise and relevant answers

In [12]:
# baseline prompt
prompt1 = ChatPromptTemplate.from_template(
"""
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
{context}

Question:
{input}

Answer:
"""
)

In [13]:
# constrained prompt
prompt2 = ChatPromptTemplate.from_template(
"""
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.
    If the answer is present, extract and summarize it clearly.
    Do NOT say "I don't know" if the answer exists in the context.
    Only say "I don't know" if the context truly does not contain the answer.

Context:
{context}

Question:
{input}

Answer:
"""
)

In [14]:

rag_chain = (
    {
        "context": semantic_retriever | build_context,
        "input": RunnablePassthrough()
    }
    | prompt1
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Which of these products is green?")

/Users/jennifertsang/miniforge3/envs/575-app/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


In [15]:
print(answer)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
Product ASIN: B08X7BCNCF
Title: Soap Colorants for Soap Making – Gel Coloring – 32 Soap Dyes Set – Bath Bomb Colorant – Slime Coloring – Bath Bomb Dye – Epoxy Resin Color Pigment
Rating: 5.0/5
Review: I am very happy with the Soap Colorants for Soap. What is especially nice about this set is that the ingredient list is made of familiar items such as sugar, potato starch and citric acid.  The pigments used in the set are also listed on the outer box. This assures me the gel pigments are body-safe. Previously, I have been using food dye for my DIY shower steamers. This is a much better option as it is much more pigments, a better color variety and a much better value. 32 colors is an excellent color palette for me. This is a product made and packaged in the Ukraine with a telephone number

In [16]:
rag_chain = (
    {
        "context": semantic_retriever | build_context,
        "input": RunnablePassthrough()
    }
    | prompt2
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Which of these products is green?")

In [17]:
print(answer)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.
    If the answer is present, extract and summarize it clearly.
    Do NOT say "I don't know" if the answer exists in the context.
    Only say "I don't know" if the context truly does not contain the answer.

Context:
Product ASIN: B08X7BCNCF
Title: Soap Colorants for Soap Making – Gel Coloring – 32 Soap Dyes Set – Bath Bomb Colorant – Slime Coloring – Bath Bomb Dye – Epoxy Resin Color Pigment
Rating: 5.0/5
Review: I am very happy with the Soap Colorants for Soap. What is especially nice about this set is that the ingredient list is made of familiar items such as sugar, potato starch and citric acid.  The pigments used in the set are also listed on the outer box. This assures me the gel pigments are body-safe. Previously, I have been using food dye for my DIY shower steamers. This is a much bett

### RAG Pipeline

#### With BM25 Retriever

In [18]:
bm25_retriever = BM25Retriever.from_documents(
        docs,  # docs is a Langchain Document objects
        k=5    # returns top 5 results
        )

# Note: LangChain BM25Retriever acutomatically tokenizes the query and documents behind the scene
 
# Retriever invoke returns the top_k docs
result = bm25_retriever.invoke(query) # query = user query

In [19]:
rag_chain = (
    {
        "context": bm25_retriever | build_context,
        "input": RunnablePassthrough()
    }
    | prompt1
    | llm
    | StrOutputParser()
)

bm25_answer = rag_chain.invoke("Which of these products is green?")

In [20]:
print(bm25_answer)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
Product ASIN: B017QU7N7Y
Title: Lion Brand Yarn 5800-526 Martha Stewart Glitter Eyelash Yarn, Brownstone
Rating: 4.0/5
Review: This pretty, sparkly eyelash yarn  adds a lovely, glamorous  look to to various projects.  The yarn has a touch of delicate pink, and also reflects sparkly rainbow colors.  I am just learning to work with eyelash yarn, but can say it adds a lovely accent to a hat, or scarf.  It can be a bit tricky to work with at first, but you will pick it up as you go along.  For me, it works better in knit projects then as a crochet yarn.  Make sure you note the weight and yardage.  These are tiny skeins, so don't be taken by surprise by that.  It is a bulky yarn, so that is another thing to consider in picking a project.  I like it.

Product ASIN: B09YM3K52Q
Title: Crafted B

#### With Hybrid Retriever

In [21]:
# Create the ensemble
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.4, 0.6]  # Example: asigning 40% importance to BM25, 60% to Semantic Search
)

# Invoke to get combined results
docs = ensemble_retriever.invoke("Sewing machine")

In [22]:
rag_chain = (
    {
        "context": ensemble_retriever | build_context,
        "input": RunnablePassthrough()
    }
    | prompt1
    | llm
    | StrOutputParser()
)

ensemble_answer = rag_chain.invoke("Green yarn")

In [23]:
print(ensemble_answer)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
Product ASIN: B07YF38F25
Title: Lion Brand Yarn Ferris Wheel Yarn, Multicolor Yarn for Knitting, Crocheting, and Crafts, 3-Pack, Cherry on Top
Rating: 1.0/5
Review: I normally buy Lion brand directly from their website.  I’ve lesrned to rewind my yarn to see what issues might crop up while knitting. This one has garbage like this all throughout it. Idk if it’s seconds Amazon bought or I just got a lousy ball.  I purchased two, but given what this looks like I’m not wasting my time rewinding the other. I’ll return these. I have 2 other colors I purchased to check next. Hoping this is just a bad lot, bc I really like the colors. Might try again later. Idk. Pretty bummed right now.

Product ASIN: B0BWSF1HPY
Title: Arm Knitting Yarn for Chunky Braided Knot Throw Blanket DIY, Soft Extra Cott

### RAG Evaluation
- Perform qualitative evalution, considering accuracy, completeness, and fluency.

In [41]:
# Query 1: "sewing machine"
ensemble_q1 = rag_chain.invoke("sewing machine")
print(ensemble_q1)

/Users/jennifertsang/miniforge3/envs/575-app/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the Product ASIN when possible.

Context:
Product ASIN: B097FPLKP2
Title: SINGER | 4423 Heavy Duty Sewing Machine & Beginners & Sewing Machine Accessory Kit, Including 9 Presser Feet, Twin Needle, and Case, Clear - Sewing Made Easy
Rating: 2.0/5
Review: I was looking for something that could sew jeans and such heavy fabrics or multiple layered fabrics. Thought this one would be great as the name itself says heavy duty. Machine is super easy to understand as well as to start using. This arrived the next day I ordered it. These are the only reasons to give these stars . When I unboxed it, I found there was a threaded spinner white bobbin inside already and a same whistle color thread spinner bobbin in the thread spool holder and needle was threaded as well. I believe no new machine comes With threads ready like this. Hence this i

In [24]:
ensemble_q2 = rag_chain.invoke("acrylic paint")
print(ensemble_q2)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
Product ASIN: B0BKPGY8D9
Title: Nicpro 32 Colors Outdoor Acrylic Paint Bulk with Brush and Sponge, Knife, Non-Toxic Paint for Multi-surface Rock, Wood, Fabric, Leather, Crafts, Canvas, Shoes and Wall Painting
Rating: 5.0/5
Review: I have plenty of acrylics.  One of the things I don't have, however, is acrylic that is designed to be used outdoors.  This kit is just that.<br /><br />HOW DOES THIS COME?<br />This set comes in a box that is pretty tough.  It has a series of brushes inside, a sponge, and a palette knife.<br /><br />PAINT CONTAINERS<br />These paint containers are flip tops which dispense paints through a small hole.<br /><br />COLORS<br />The colors in this set vary from bright colors, to several metallics.  There are two containers of white and one of black which is good fo

In [25]:
ensemble_q3 = rag_chain.invoke("Singer 5532")
print(ensemble_q3)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
Product ASIN: B00WI1L58A
Title: SINGER 57261 Vintage Sewing Baskets, Large, Pink/Black
Rating: 5.0/5
Review: My teenager "adopted" my sewing basket. So when i started looking for a new one, i saw the name Singer, and i knew it would be quality. I was right! Bonus! It even came with items as a starter kit! Lots of room for small and large items, and the fabric is adorable. Ill have this a long time, or at least till my 7 year old turns 16 and "adopts" this one, lol.

Product ASIN: B0092RC3YI
Title: SINGER 5532 Heavy Duty Extra-High Sewing Speed Portable Sewing Machine with Metal Frame and Stainless Steel Bedplate
Rating: 5.0/5
Review: The SINGER 5532 HEAVY DUTY SEWING MACHINE is a high-speed machine that sews 1,100 stitches per minute.  It's a lot faster than my ancient Singer "Touch & S

In [26]:
ensemble_q4 = rag_chain.invoke("Animal stickers")
print(ensemble_q4)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
Product ASIN: B089S7H2F8
Title: Knaid Butterfly Dragonfly Insects Stickers Set (240 Pieces) - PET Transparent Waterproof Decorative Decals for Scrapbook DIY Crafts Album Bullet Journal Planner Water Bottles Phone Cases Laptops
Rating: 5.0/5
Review: Beautiful stickers very sheer and great results

Product ASIN: B09TXV5N83
Title: Aromoty Flower Gold foil Holographic Stickers Set( 120 pieces with 3 Themes)-Resin Transparent Waterproof Stickers,Butterfly,Vintage Daily Planner Stickers for Scrapbook Junk Bullet Journals, Laptop,Water bottles Decor for kids adults (Flower B- Warm)
Rating: 5.0/5
Review: Lots of pretty shiny stickers and great quality, variety.

Product ASIN: B09PMZJ49B
Title: 120 Pieces Vintage Scrapbooking Stickers DIY Journaling Scrapbook Adhesive Washi Paper Stamp Stickers 

In [27]:
ensemble_q5 = rag_chain.invoke("Purple gel pen")
print(ensemble_q5)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
Product ASIN: B01N0D2SHX
Title: Gel Pens,Tanmit Gel Pens Set, 120 Colored Gel Pen plus 120 Refills for Adults Coloring Books, Drawing, Art Projects (No Duplicates)
Rating: 1.0/5
Review: I got this set and was excited to use them.  The pens worked at first.  Only at first.  Each pen has run dry with minimal use. By minimal, I mean a few written lines at most. I've used the replacement cartridges for several immediately, but they are made with the same crappy manufacturing.<br /><br />It seems like a steal with so many colors, but it's not. They are taking your money and the pens are literally empty except for a thin layer of dried ink coating the wall of the pen shaft.  You can look inside each one and see they are coated inside to look full but are empty.

Product ASIN: B09D2SFW5Y
Title

In [28]:
ensemble_q6 = rag_chain.invoke("cute embroidery patterns")
print(ensemble_q6)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
Product ASIN: B099VZK5BW
Title: Stitcher's Revolution Iron-On Transfer Pattern for Embroidery, Around The World Landmarks
Rating: 5.0/5
Review: These iron on transfers take me back to childhood when I was first learning to stitch.  They are easy to apply and work up quickly.  They make it easy to personalize all sorts of cloth items, clothing linens, bags, etc.  And, if you don't stitch, permanent markers also work well. (Whatever you choose, the item must be able to withstand a hot iron.)  I am using this set to make a quilt for my guest room.  It should bring a smile to my visitors faces.  And, since you can use each transfer several times, it is a good deal for the price.  There all sorts of designs available, so you can easily find one to suit your needs.

Product ASIN: B09PYHCFMV
T

In [29]:
ensemble_q7 = rag_chain.invoke("yarn made of natural fibers")
print(ensemble_q7)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
Product ASIN: B09YM3K52Q
Title: Crafted By Catherine Velvet Chain Yarn - 3 Pack, Red, Gauge 6 Super Bulky
Rating: 5.0/5
Review: Very beautiful yarn with an oddly dense/firm feel. Looks good worked up and was a nice, easy yarn to work with. A couple of my skeins had knots, one didn't. Overall, a good yarn and I would recommend it. Keep in mind, it has little to no drape really, though using larger needles/hooks could help with that some.

Product ASIN: B0BWSF1HPY
Title: Arm Knitting Yarn for Chunky Braided Knot Throw Blanket DIY, Soft Extra Cotton Washable Tube Bulky Giant Yarn for Weave Craft Crochet (Dark Gray 2.2lb)
Rating: 5.0/5
Review: [[VIDEOID:21dfc200bc65819023d8994064a70121]] Seems like simple and normal yarn.  The feel was pretty soft I thought.<br /><br />Color seems accurate 

In [32]:
ensemble_q8 = rag_chain.invoke("best paint for wood")
print(ensemble_q8)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
Product ASIN: B08KHSGR3C
Title: Moshify Pebeo Vitrea 160 Glass Paint - Transparent Colour Glossy and Brilliant - Made in France - DIY Craft Paint for Stained Glass and More - Set of 8 Colors Bundled Application Brushes
Rating: 5.0/5
Review: I have used glass paints in the past, but none were ever permanent. I like the Pebeo Vitrea 160 Glass Paint set with the Moshify brushes because this is a good variety of colors to start. I'm currently working on a project with them. I'm painting a clear glass mug. The Moshify brushes are nice, even and have a secure ferrule. They should last a very long time with good maintenance. To use as paint, it's just like any other paint where it can be layered and the colors blend well. The difference is that this is a heat-set paint. The paint is set once y

In [31]:
ensemble_q9 = rag_chain.invoke("what's the best type of yarn to make clothes with")
print(ensemble_q9)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
Product ASIN: B0BWSF1HPY
Title: Arm Knitting Yarn for Chunky Braided Knot Throw Blanket DIY, Soft Extra Cotton Washable Tube Bulky Giant Yarn for Weave Craft Crochet (Dark Gray 2.2lb)
Rating: 5.0/5
Review: [[VIDEOID:21dfc200bc65819023d8994064a70121]] Seems like simple and normal yarn.  The feel was pretty soft I thought.<br /><br />Color seems accurate to the pictures.  Felt like what you’d use to make sweaters or something similar.

Product ASIN: B017QU7N7Y
Title: Lion Brand Yarn 5800-526 Martha Stewart Glitter Eyelash Yarn, Brownstone
Rating: 4.0/5
Review: This pretty, sparkly eyelash yarn  adds a lovely, glamorous  look to to various projects.  The yarn has a touch of delicate pink, and also reflects sparkly rainbow colors.  I am just learning to work with eyelash yarn, but can say i

In [33]:
ensemble_q10 = rag_chain.invoke("what is a good type of yarn for a beginner")
print(ensemble_q10)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

Context:
Product ASIN: B017QU7N7Y
Title: Lion Brand Yarn 5800-526 Martha Stewart Glitter Eyelash Yarn, Brownstone
Rating: 4.0/5
Review: This pretty, sparkly eyelash yarn  adds a lovely, glamorous  look to to various projects.  The yarn has a touch of delicate pink, and also reflects sparkly rainbow colors.  I am just learning to work with eyelash yarn, but can say it adds a lovely accent to a hat, or scarf.  It can be a bit tricky to work with at first, but you will pick it up as you go along.  For me, it works better in knit projects then as a crochet yarn.  Make sure you note the weight and yardage.  These are tiny skeins, so don't be taken by surprise by that.  It is a bulky yarn, so that is another thing to consider in picking a project.  I like it.

Product ASIN: B0BWSF1HPY
Title: Arm Knitt